# NB_CoreReporting_BronzeToSilver

Core reporting entities: Bronze → Silver (**full overwrite**). One cell per table. Run **reference** normalization tables before `core_country` / `core_okta_user`; run core silvers before `dim_project`.


In [ ]:
# Workspace folder: C:\spark-dev\CoE_transformation_framework (see README)
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
REBUILD_MEDALLION_CACHE = True
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "75000")
from Common.io import sync_rebuild_medallion_cache_env

sync_rebuild_medallion_cache_env(REBUILD_MEDALLION_CACHE)


In [ ]:
%run ./Common/bootstrap


## `core_asset`

In [ ]:
apply_local_merge_spark_conf(spark)

bronze = read_table(TABLE_CORE_ASSET_BRONZE)
silver = build_core_asset_silver(bronze)
write_full_table(silver, TABLE_CORE_ASSET_SILVER)
show_sample(silver.limit(10))


## `core_country`

Requires `normalization_countries_core` Silver (run reference notebook first).

In [ ]:
norm_core = read_table(TABLE_NORMALIZATION_COUNTRIES_CORE_SILVER)
bronze = read_table(TABLE_CORE_COUNTRY_BRONZE)
silver = build_core_country_silver(bronze, norm_core)
write_full_table(silver, TABLE_CORE_COUNTRY_SILVER)
show_sample(silver.limit(10))


## `core_billing`

In [ ]:
bronze = read_table(TABLE_CORE_BILLING_BRONZE)
silver = build_core_billing_silver(bronze)
write_full_table(silver, TABLE_CORE_BILLING_SILVER)
show_sample(silver.limit(10))


## `core_project`

In [ ]:
bronze = read_table(TABLE_CORE_PROJECT_BRONZE)
silver = build_core_project_silver(bronze)
write_full_table(silver, TABLE_CORE_PROJECT_SILVER)
show_sample(silver.limit(10))


## `core_okta_user`

Requires `normalization_countries_okta` Silver.

In [ ]:
norm_okta = read_table(TABLE_NORMALIZATION_COUNTRIES_OKTA_SILVER)
bronze = read_table(TABLE_CORE_OKTA_USER_BRONZE)
silver = build_core_okta_user_silver(bronze, norm_okta)
write_full_table(silver, TABLE_CORE_OKTA_USER_SILVER)
show_sample(silver.limit(10))


## `dim_project`

Silver → Silver enrich (run `core_project`, `core_billing`, `core_country` first).

In [ ]:
core_project = read_table(TABLE_CORE_PROJECT_SILVER)
core_billing = read_table(TABLE_CORE_BILLING_SILVER)
core_country = read_table(TABLE_CORE_COUNTRY_SILVER)
silver = build_dim_project_silver(core_project, core_billing, core_country)
write_full_table(silver, TABLE_DIM_PROJECT_SILVER)
show_sample(silver.limit(10))


## `core_use_case_implementation`

In [ ]:
bronze = read_table(TABLE_CORE_USE_CASE_IMPLEMENTATION_BRONZE)
silver = build_core_use_case_implementation_silver(bronze)
write_full_table(silver, TABLE_CORE_USE_CASE_IMPLEMENTATION_SILVER)
show_sample(silver.limit(10))


## `core_use_case_adjustment`

In [ ]:
bronze = read_table(TABLE_CORE_USE_CASE_ADJUSTMENT_BRONZE)
silver = build_core_use_case_adjustment_silver(bronze)
write_full_table(silver, TABLE_CORE_USE_CASE_ADJUSTMENT_SILVER)
show_sample(silver.limit(10))
